<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/12_neural_networks_that_write_like_shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 12: Neural Networks That Write Like Shakespeare - Recurrent Layers for Variable-Length Data**

This notebook summarizes Chapter 12 of *Grokking Deep Learning*:

1. The Challenge of Arbitrary Length
2. Do Comparisons Really Matter?
3. The Surprising Power of Averaged Word Vectors
4. How Is Information Stored in These Embeddings?
5. How Does a Neural Network Use Embeddings?
6. The Limitations of Bag-of-Words Vectors
7. Using Identity Vectors to Sum Word Embeddings
8. Matrices That Change Absolutely Nothing
9. Learning the Transition Matrices
10. Learning to Create Useful Sentence Vectors
11. Forward Propagation in Python
12. How Do You Backpropagate into This?
13. Let's Train It!
14. Setting Things Up
15. Forward Propagation with Arbitrary Length
16. Backpropagation with Arbitrary Length
17. Weight Update with Arbitrary Length
18. Execution and Output Analysis
19. Summary

Chapter 11 taught a network to build **word embeddings**, vectors that capture the meaning of a single word. This chapter asks how the same idea can be stretched to **whole phrases and sentences of any length**. It compares simple ways of turning a sequence of word vectors into one fixed-size vector, points out the weakness of the simplest one (it ignores word order), and then develops a **linear recurrent neural network** that learns how order should influence the sentence vector. The network is trained on a toy task, predicting the next word, and its forward propagation, backpropagation, and weight updates are implemented for sequences of arbitrary length. The author insists that this chapter and Chapter 11 are intertwined, so the concepts of Chapter 11 should be mastered first.

The chapter opens with a quotation from Andrej Karpathy's essay on recurrent neural networks, stating that there is something magical about them.

---
## **1. The Challenge of Arbitrary Length**

**Let us model arbitrarily long sequences of data with neural networks.**

Chapter 11 introduced natural language processing (NLP): how a loss function can be modified so that a specific pattern of information is learned inside the weights of a network, and what a word embedding is and how it expresses shades of similarity to other embeddings. Here, that intuition (an embedding that conveys the meaning of *one word*) is extended to embeddings that convey the meaning of **variable-length phrases and sentences**.

Suppose we want a single vector that holds an entire sequence of symbols, just as a word embedding holds information about one word. The simplest option is to **concatenate (stack) the word embeddings**, so that the resulting vector contains the whole sequence.

> Concatenation: the sentence 'the cat sat' gives a vector three word-vectors long, while 'the cat sat still' gives one four word-vectors long
>
> <img src="pictures/chapter 12/picture 12.1.png" alt="Figure 12.1" width="600"/>

This has a clear drawback: **different sentences produce vectors of different lengths**, which makes two vectors hard to compare, because one of them sticks out at the side. The sentences "the cat sat" and "the cat sat still" should be very similar, yet the shorter vector forces a choice of which part of the longer one to compare against:

- If the vectors are aligned on the **left**, they look identical (ignoring that "the cat sat still" is, in fact, a different sentence).
- If they are aligned on the **right**, they look extraordinarily different, even though three quarters of the words are the same and in the same order.

The naive approach shows some promise, but it is far from an ideal way to represent the meaning of a sentence in a form that can be compared with other vectors.

---
## **2. Do Comparisons Really Matter?**

**Why should one care whether two sentence vectors can be compared?**

Comparing two vectors is useful because it approximates **what the neural network sees**. We cannot read the vectors, but we can tell whether they are similar or different (with the similarity function from Chapter 11). If the method that builds sentence vectors does not reflect the similarity we observe between two sentences, the network will also struggle to notice that the sentences are similar, since vectors are all it has to work with.

The point of evaluating different ways to compute sentence vectors is to adopt the **perspective of the network**. The question is whether the correlation summarization will find a correlation between sentence vectors similar to a given one and a desirable label, or whether two almost identical sentences will produce wildly different vectors, so that there is little correlation between vectors and labels. Sentence vectors should be useful for predicting things about the sentence, and at the very least **similar sentences must produce similar vectors**.

Concatenation fails here because of the arbitrary way the vectors have to be aligned. The next-simplest approach is to **take the vector of every word in the sentence and average them**.

> Matrix row average: the word vectors of a sentence are averaged into one sentence vector with the same length as a single word vector
>
> <img src="pictures/chapter 12/picture 12.2.png" alt="Figure 12.2" width="600"/>

Averaging has several attractive properties:

- There is **no alignment problem**, because every sentence vector has the same length.
- "The cat sat" and "the cat sat still" get similar vectors, because the words that go into them are similar.
- "A dog walked" will probably be similar to "the cat sat" even though no words overlap, because the words involved are themselves similar.

Averaging word embeddings is a surprisingly effective way of building sentence embeddings. It is not perfect (as the next sections show), but it captures what look like complex relationships between words. Before moving on, the author recommends taking the embeddings of Chapter 11 and experimenting with the averaging strategy.

---
## **3. The Surprising Power of Averaged Word Vectors**

**It is the amazingly powerful go-to tool for neural prediction.**

The averaged sentence vectors are tried out with the embeddings of Chapter 11: the embeddings are trained on the IMDB corpus as before, and then used to build average sentence embeddings. The listing below does the following:

- It repeats the **normalization** applied earlier when word embeddings were compared, but this time all embeddings are **prenormalized** into a matrix called `normed_weights`.
- The function `make_sent_vect` converts a review (a list of words) into an embedding by **averaging** the normalized vectors of its words (words that are not in the vocabulary are skipped). The results for all reviews are stored in the matrix `reviews2vectors`.
- The function `most_similar_reviews` takes a review, builds its vector, and computes the **dot product** with the vector of every review in the corpus. It returns the beginnings (first 40 characters) of the three highest-scoring reviews. This dot-product similarity is the one briefly discussed in Chapter 4, when learning to predict with multiple inputs.

*Note on the data:* this listing builds on **Chapter 11**. It needs the variables `weights_0_1` (the trained embedding matrix), `word2index`, `tokens` (the tokenized reviews), `raw_reviews`, and `Counter`, which exist only after the Chapter 11 code has been run on the IMDB reviews. Because every chapter has its own notebook, none of these variables exist here, so the listing is shown as **text** below instead of as a code cell. Run it at the end of the Chapter 11 notebook (in the same session) if you wish to try it. None of the code cells that follow depend on it.

```python
import numpy as np
norms = np.sum(weights_0_1 * weights_0_1,axis=1)
norms.resize(norms.shape[0],1)
normed_weights = weights_0_1 * norms

def make_sent_vect(words):
  indices = list(map(lambda x:word2index[x],\
        filter(lambda x:x in word2index,words)))
  return np.mean(normed_weights[indices],axis=0)

reviews2vectors = list()
for review in tokens:
  reviews2vectors.append(make_sent_vect(review))
reviews2vectors = np.array(reviews2vectors)

def most_similar_reviews(review):
  v = make_sent_vect(review)
  scores = Counter()
  for i,val in enumerate(reviews2vectors.dot(v)):
    scores[i] = val
  most_similar = list()

  for idx,score in scores.most_common(3):
    most_similar.append(raw_reviews[idx][0:40])
  return most_similar
most_similar_reviews(['boring','awful'])
```

When the query is the average of the two words "boring" and "awful", the three most similar reviews are all **very negative**. The book prints the beginnings of the reviews returned: "I am amazed at how boring this film", "This is truly one of the worst dep", and "It just seemed to go on and on and". Perhaps surprisingly, the vectors therefore contain interesting statistical information, such that **negative and positive embeddings cluster together**.

---
## **4. How Is Information Stored in These Embeddings?**

**When word embeddings are averaged, the average shapes remain.**

This section asks for a little abstract thought, and the author recommends digesting it over some time. The idea is to see a word vector not as a list of numbers but as a **squiggly line**, with high and low points that correspond to the high and low values at different positions of the vector.

> A word vector drawn as a squiggly line: its height follows the nine values (the first value is printed as 2 in the book, presumably .2)
>
> <img src="pictures/chapter 12/picture 12.3.png" alt="Figure 12.3" width="600"/>

If several words are selected from the corpus, each has its own, unique shape. Yet similar words have similar shapes.

> Squiggles of four words: 'terrible' and 'boring' resemble each other, and so do 'wonderful' and 'beautiful', but the two pairs differ from one another
>
> <img src="pictures/chapter 12/picture 12.4.png" alt="Figure 12.4" width="600"/>

If these little squiggles were clustered, words with similar meaning would end up together. More importantly, **parts of the squiggles carry meaning by themselves**. For the negative words, for instance, there is a downward-then-upward spike at about 40% from the left, and it would keep showing up as more words were drawn. There is nothing magical about that spike: if the network were retrained, it would probably appear somewhere else. It signals negativity **only because all the negative words have it**.

During training, these shapes are molded so that different curves in different positions convey meaning (as discussed in Chapter 11). When the curves of the words in a sentence are averaged, the **most dominant meanings of the sentence survive, and the noise contributed by any single word averages away**.

---
## **5. How Does a Neural Network Use Embeddings?**

**Neural networks detect the curves that correlate with a target label.**

Embeddings can be viewed as squiggly lines with distinctive curves, and these curves are developed during training to accomplish the target objective. Words that are similar in some way often share a distinctive bend: a combination of high and low values among the weights. The question now is how the correlation summarization processes these curves as input.

A neural network consumes embeddings just as it consumed the streetlight dataset in the early chapters: it looks for **correlation between the bumps and curves of the hidden layer and the target label** it is trying to predict. That is why words with one particular aspect of similarity share similar bumps and curves. During training, the network develops unique characteristics that tell words apart, and groups words together (giving them similar bumps and curves) when that helps to make accurate predictions. This is another way of summarizing the lessons from the end of Chapter 11.

Going further, the chapter considers what it means to **sum these embeddings into a sentence embedding**. Taking an average across the word embeddings of a sentence gives a vector with the average characteristics of its words: with many positive words, the final embedding looks somewhat positive, and the noise of the other words largely cancels. The approach is, however, a bit mushy, because given enough words **the different wavy lines average out to roughly a straight line**.

This is the **first weakness** of the approach. When an arbitrarily long sequence is stored in a fixed-length vector, storing too much makes the sentence vector (an average of many word vectors) converge to a straight line, a vector of near-zeros. The storage of information **does not decay nicely**: trying to store too many words leaves almost nothing. A sentence is often not that long, though, and if it contains repeating patterns, the sentence vector keeps the dominant patterns present across the summed word vectors (such as the negative spike of the previous section), so such vectors can still be useful.

---
## **6. The Limitations of Bag-of-Words Vectors**

**Order becomes irrelevant when word embeddings are averaged.**

The biggest problem of average embeddings is that they have **no concept of order**. The sentences "Yankees defeat Red Sox" and "Red Sox defeat Yankees" produce identical average vectors, although they convey exactly opposite information. Grammar and syntax are ignored too, so "Sox Red Yankees defeat" yields the same embedding as well.

Summing or averaging word embeddings to form the embedding of a phrase is classically called a **bag-of-words** approach: as with words thrown into a bag, the order is not preserved. Any sentence can be scrambled and will give the same vector, because **addition is associative** ($a + b = b + a$).

The real topic of the chapter is generating sentence vectors **in a way where order does matter**, so that scrambling the words changes the vector. Moreover, *the way in which order changes the vector* should be **learned**: the network's representation of order is then shaped by a task in language and, hopefully, captures the essence of order in language. Language is merely a challenging yet universally known example, and the statements generalize to any sequence.

One of the most famous and successful ways to generate vectors for sequences is the **recurrent neural network (RNN)**. To show how it works, the chapter starts from a new and seemingly wasteful way of computing the average of word embeddings, based on the **identity matrix**.

### **6.1 The Identity Matrix**

An identity matrix is an arbitrarily large **square** matrix (as many rows as columns) filled with 0s, with 1s running from the top-left corner to the bottom-right corner. The book shows three examples, of sizes 2 x 2, 3 x 3, and 4 x 4:

$$\begin{bmatrix}1&0\\0&1\end{bmatrix} \qquad \begin{bmatrix}1&0&0\\0&1&0\\0&0&1\end{bmatrix} \qquad \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&1&0\\0&0&0&1\end{bmatrix}$$

An identity matrix has a single purpose: **vector-matrix multiplication of any vector with it returns the original vector**. For example, multiplying the vector `[3,5]` by the first matrix gives `[3,5]`.

---
## **7. Using Identity Vectors to Sum Word Embeddings**

**The same logic can be implemented with a different approach.**

An operation that returns its own input may look useless. Here, however, it is used as a **teaching tool**: it sets up a more elaborate way of summing word embeddings, in which the network can take word order into account when it builds the final sentence embedding.

> The standard technique: the word vectors of 'Red Sox defeat Yankees' are simply added (dividing by the number of words would give the average)
>
> <img src="pictures/chapter 12/picture 12.5.png" alt="Figure 12.5" width="600"/>

> The same sum with a step between the additions: each running result is multiplied (vector-matrix) by an identity matrix before the next word vector is added
>
> <img src="pictures/chapter 12/picture 12.6.png" alt="Figure 12.6" width="600"/>

The first figure shows the standard way of summing several word embeddings into a sentence embedding. The second inserts a step **between** the additions. The vector for "Red" is multiplied by an identity matrix, and the output is added to the vector for "Sox"; that sum is multiplied by the identity matrix and added to the vector for "defeat"; and so on to the end of the sentence.

Since multiplying by the identity matrix returns the same vector, the second process produces **exactly the same sentence embedding** as the first. The computation is wasteful, but that is about to change. The key observation is that if the matrices were **anything other than the identity matrix**, then changing the order of the words would change the resulting embedding. The next section verifies the identity-matrix version in Python.

---
## **8. Matrices That Change Absolutely Nothing**

**Sentence embeddings can be created with identity matrices in Python.**

The following cells play with identity matrices in NumPy and implement the new sentence-vector technique of the previous section, proving that it yields identical sentence embeddings.

### **8.1 Four Vectors and an Identity Matrix**

The first cell creates four vectors of length 3 (`a`, `b`, `c`, and `d`) and a 3 x 3 identity matrix with `np.eye(3)`. Identity matrices are always square, and any square matrix with 1s on the **diagonal** (from top-left to bottom-right) and 0s elsewhere is an identity matrix. Printing it shows the matrix with 1.0 on the diagonal and 0.0 elsewhere.

In [1]:
import numpy as np

a = np.array([1,2,3])
b = np.array([0.1,0.2,0.3])
c = np.array([-1,-0.5,0])
d = np.array([0,0,0])

identity = np.eye(3)
print(identity)

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


### **8.2 Vector-Matrix Multiplication with the Identity Matrix**

Each of the four vectors is now multiplied by the identity matrix with NumPy's `dot` function. The output is in every case a vector identical to the input: `[1. 2. 3.]`, `[0.1 0.2 0.3]`, `[-1. -0.5 0.]`, and `[0. 0. 0.]`.

In [2]:
print(a.dot(identity))
print(b.dot(identity))
print(c.dot(identity))
print(d.dot(identity))

[1. 2. 3.]
[0.1 0.2 0.3]
[-1.  -0.5  0. ]
[0. 0. 0.]


### **8.3 Sentence Embeddings with and without the Identity Matrix**

Because the multiplication returns the vector it started with, incorporating it into the sentence embedding is trivial. In the cell below, three toy word vectors (`this`, `movie`, and `rocks`) are combined in two ways: by plain addition, and by the identity-matrix procedure (multiply the first vector by the identity, add the second, multiply again, add the third). Both prints give the same sentence embedding, `[13 15 17]` (the second one is displayed with decimal points, `[13. 15. 17.]`, because the identity matrix holds floating-point numbers).

In [3]:
this = np.array([2,4,6])
movie = np.array([10,10,10])
rocks = np.array([1,1,1])

print(this + movie + rocks)
print((this.dot(identity) + movie).dot(identity) + rocks)

[13 15 17]
[13. 15. 17.]


This equality holds only because the identity matrix is such a special matrix. It is in fact **the only matrix guaranteed to return the same vector it is multiplied with**; no other matrix has this guarantee. The obvious next question is what happens if a different matrix is used.

---
## **9. Learning the Transition Matrices**

**What if the identity matrices were allowed to change in order to minimize the loss?**

The goal is to generate sentence embeddings that **cluster according to the meaning of the sentence**, so that the vector of a sentence can be used to find sentences with a similar meaning, and that **care about the order of the words**. Summing word embeddings gave "Red Sox defeat Yankees" and "Yankees defeat Red Sox" the same vector despite their opposite meanings. We want these two sentences to receive **different** embeddings that still cluster in a meaningful way. The theory is that the identity-matrix construction, with any matrix other than the identity, produces embeddings that depend on the order.

Which matrix should replace the identity? There are infinitely many choices, but deep learning has a standard answer: **learn the matrix like any other matrix of a network**. Learning requires a task. Here, the task must force the network to create interesting sentence embeddings by learning both **useful word vectors and useful modifications of the identity matrices**.

> Supervised learning: use what you know to predict what you want to know
>
> <img src="pictures/chapter 12/picture 12.7.png" alt="Figure 12.7" width="600"/>

The goal resembles the one used to obtain word embeddings (filling in a blank). The chosen task is similarly simple: train a network to take a **list of words and predict the next word**.

> The task: given the words 'This', 'movie', 'was', the network predicts the next word, here 'great'
>
> <img src="pictures/chapter 12/picture 12.8.png" alt="Figure 12.8" width="600"/>


---
## **10. Learning to Create Useful Sentence Vectors**

**Create the sentence vector, make a prediction, and modify the sentence vector through its parts.**

For this experiment, the network should not be thought of like earlier networks. Instead: **create a sentence embedding, use it to predict the next word, and then modify the parts that formed the embedding** so that the prediction becomes more accurate. Since the next word is being predicted, the embedding is built only from the part of the sentence seen so far.

> The network for the input 'Red Sox defeat' and the target 'Yankees': word vectors joined by identity matrices form the sentence embedding, and a softmax over the vocabulary predicts the next word
>
> <img src="pictures/chapter 12/picture 12.9.png" alt="Figure 12.9" width="600"/>

The network has two steps: it **creates the sentence embedding**, and then uses it to **predict which word comes next** (a softmax over the whole vocabulary; the target is the one-hot vector of "Yankees"). The matrices drawn between the word vectors are written as "Identity matrix" because they only **start** as identity matrices. During training, gradients are backpropagated into them and they are updated to make better predictions, like the rest of the weights.

In this way the network learns to incorporate **more information than a plain sum of word embeddings**. Once the matrices are free to become non-identity matrices, the network can learn embeddings in which the order of presentation changes the sentence embedding. This change is not arbitrary: the network learns to represent order in the way that is **useful for predicting the next word**.

The **transition matrices** (the matrices that start as identity matrices) are also **constrained to be the same matrix**. The matrix used to go from "Red" to "Sox" is reused to go from "Sox" to "defeat". Whatever logic is learned in one transition is reused in the next, so only logic that is useful at **every** predictive step can be learned.

---
## **11. Forward Propagation in Python**

**A simple forward propagation of this idea, as a toy version.**

The first cell sets up the weights for a toy vocabulary of nine words. It creates three sets of weights:

- `word_vects`: a Python dictionary of **word embeddings**, here all-zero vectors of length 3 (one row each);
- `identity`: the **transition matrix**, a 3 x 3 identity matrix;
- `sent2output`: the **classification layer**, a 3 x 9 matrix of random numbers that predicts the next word from a sentence vector of length 3.

The function `softmax` is also defined (here for 2-dimensional input).

In [4]:
import numpy as np

def softmax(x_):
    x = np.atleast_2d(x_)
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

word_vects = {}
word_vects['yankees'] = np.array([[0.,0.,0.]])
word_vects['bears'] = np.array([[0.,0.,0.]])
word_vects['braves'] = np.array([[0.,0.,0.]])
word_vects['red'] = np.array([[0.,0.,0.]])
word_vects['sox'] = np.array([[0.,0.,0.]])
word_vects['lose'] = np.array([[0.,0.,0.]])
word_vects['defeat'] = np.array([[0.,0.,0.]])
word_vects['beat'] = np.array([[0.,0.,0.]])
word_vects['tie'] = np.array([[0.,0.,0.]])

sent2output = np.random.rand(3,len(word_vects))

identity = np.eye(3)

Forward propagation is now trivial. For the sentence "red sox defeat" (target: "yankees"), the cell below builds a **sentence embedding** by adding the embedding of each word after multiplying the previous result by the identity matrix, and then **predicts over the whole vocabulary** with a softmax. Because all word embeddings are still zero vectors, the prediction is uniform: all nine probabilities are 0.1111 (one ninth), as printed in the book.

In [5]:
layer_0 = word_vects['red']
layer_1 = layer_0.dot(identity) + word_vects['sox']
layer_2 = layer_1.dot(identity) + word_vects['defeat']

pred = softmax(layer_2.dot(sent2output))
print(pred)

[[0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
  0.11111111 0.11111111 0.11111111]]


---
## **12. How Do You Backpropagate into This?**

**It may look trickier, but the steps are the ones already learned.**

At first it may not be clear how backpropagation works for this network, but it is simple. The forward code consists of familiar pieces:

> The forward propagation code divided into its parts: the first term is a normal network (chapters 1-5), the added word vectors are 'some sort of strange additional piece', and the softmax prediction is a normal network again (Chapter 9)
>
> <img src="pictures/chapter 12/picture 12.10.png" alt="Figure 12.10" width="600"/>

From the previous chapters, computing a loss and backpropagating to the gradient at `layer_2` (called `layer_2_delta`) should feel comfortable. The question is **which direction to backpropagate in**: backward through the identity-matrix multiplication to `layer_1`, or into `word_vects['defeat']`. The answer follows from a rule: **when two vectors are added in forward propagation, the same gradient is backpropagated into both sides of the addition**. So `layer_2_delta` is backpropagated twice: once across the identity matrix to create `layer_1_delta`, and once to `word_vects['defeat']`.

The cell below does the whole backward pass and the weight update for the toy example. Its annotations in the book explain three details:

- `y` is the **one-hot target vector for "yankees"**, and `pred_delta = pred - y` is the output gradient.
- `defeat_delta = layer_2_delta * 1` and `sox_delta = layer_1_delta * 1`: the multiplication by 1 can be ignored, as in Chapter 11.
- The identity matrix is used twice in the forward pass, so it is updated **twice**, each time with the outer product of the vector that entered the multiplication (`layer_0`, then `layer_1`) and the delta that came out of it. The classification layer `sent2output` is updated with the outer product of `layer_2` and `pred_delta`. The step size is `alpha = 0.01`.

In [6]:
y = np.array([1,0,0,0,0,0,0,0,0])

pred_delta = pred - y
layer_2_delta = pred_delta.dot(sent2output.T)
defeat_delta = layer_2_delta * 1
layer_1_delta = layer_2_delta.dot(identity.T)
sox_delta = layer_1_delta * 1
layer_0_delta = layer_1_delta.dot(identity.T)
alpha = 0.01
word_vects['red'] -= layer_0_delta * alpha
word_vects['sox'] -= sox_delta * alpha
word_vects['defeat'] -= defeat_delta * alpha
identity -= np.outer(layer_0,layer_1_delta) * alpha
identity -= np.outer(layer_1,layer_2_delta) * alpha
sent2output -= np.outer(layer_2,pred_delta) * alpha

---
## **13. Let's Train It!**

**All the tools are available, so the network is trained on a toy corpus.**

To build intuition, the new network is first trained on a toy task using the **bAbI dataset**, a synthetically generated question-answer corpus meant to teach machines to answer simple questions about an environment. It is not used for question answering yet, but its simplicity makes it easier to see the effect of **learning the identity matrix**. The bAbI archive is downloaded and unpacked with these bash commands from the book (they are shell commands, not Python):

```
wget http://www.thespermwhale.com/jaseweston/babi/tasks_1-20_v1-1.tar.gz
tar -xvf tasks_1-20_v1-1.tar.gz
```

*Note on the data:* the following code expects the file `tasksv11/en/qa1_single-supporting-fact_train.txt` from this archive, in the folder where the notebook runs (the commands above create the folder `tasksv11` there). The address above may no longer respond, so the setup cell below, which is **not part of the book's code**, downloads just this one file from the book's public repository instead; in that case the two commands above are not needed. The code opens the file and keeps the first 1000 lines, each **lowercased, stripped of its newline, split into words, and without the first element** (the line number). The cells in the rest of the chapter depend on this cell and on its variable `tokens`.

**Setup (not from the book).** The next cell downloads the bAbI file used by the code that follows.

In [7]:
# Setup (not from the book): download the bAbI file
!wget -q -nc -P tasksv11/en https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/master/tasksv11/en/qa1_single-supporting-fact_train.txt

In [8]:
import sys,random,math
from collections import Counter
import numpy as np

f = open('tasksv11/en/qa1_single-supporting-fact_train.txt','r')
raw = f.readlines()
f.close()

tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n","").split(" ")[1:])

print(tokens[0:3])

[['mary', 'moved', 'to', 'the', 'bathroom.'], ['john', 'went', 'to', 'the', 'hallway.'], ['where', 'is', 'mary?', '\tbathroom\t1']]


The book prints the beginning of `tokens`: three short word lists, `['Mary', 'moved', 'to', 'the', 'bathroom']`, `['John', 'went', 'to', 'the', 'hallway']`, and `['Where', 'is', 'Mary', 'bathroom']` (the capital letters shown there do not match the lowercasing in the code, which would give lowercase words).

The dataset contains simple **statements and questions** (with punctuation removed), and each question is followed by its correct answer. In a QA setting, a network reads the statements in order and answers the questions, correctly or not, on the basis of the most recently read statements. In this chapter the network is instead trained to **finish each sentence when it is given one or more starting words**. Along the way, the importance of letting the **recurrent matrix** (formerly the identity matrix) learn becomes visible.

---
## **14. Setting Things Up**

**Before matrices can be created, the number of parameters must be known.**

As for the word-embedding network of Chapter 11, a few counts, lists, and utility functions are needed for the predict, compare, and learn process:

- a list `vocab` of the vocabulary words and a lookup dictionary `word2index` that converts between a word and its position (the index selects the row or column of the embedding and prediction matrices that belongs to the word);
- `words2indices`, which converts a list of words into a list of indices;
- `softmax`, used to predict the next word (this version subtracts the maximum before exponentiating and replaces the 2-dimensional one used in the toy example).

In [9]:
vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)
vocab = list(vocab)

word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i

def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

The next cell initializes the random seed (for consistent results), sets the embedding size to 10, and creates the following weights, as annotated in the book:

| Variable | Role |
|---|---|
| `embed` | the **word embeddings**, one row per vocabulary word, small random values |
| `recurrent` | the **embedding-to-embedding** matrix, initially the identity matrix (it will be learned) |
| `start` | the sentence embedding of an **empty sentence**, initially zeros; it is key for modeling how sentences tend to start |
| `decoder` | the **embedding-to-output** weights, as with embeddings in Chapter 11 |
| `one_hot` | one-hot lookups, used for the loss function |

In [10]:
np.random.seed(1)
embed_size = 10

embed = (np.random.rand(len(vocab),embed_size) - 0.5) * 0.1

recurrent = np.eye(embed_size)

start = np.zeros(embed_size)

decoder = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1

one_hot = np.eye(len(vocab))

---
## **15. Forward Propagation with Arbitrary Length**

**Forward propagation follows the same logic as described earlier.**

The function `predict` forward propagates and predicts the next word. Its construction may feel unfamiliar, but the procedure is the same as before: embeddings are summed, with the identity matrix replaced by the matrix `recurrent`. The book's text says that this matrix is initialized to all 0s, but the printed code of the previous cell initializes it with `np.eye(embed_size)`, the identity matrix; the notebook follows the code.

A second difference is that a prediction (`layer['pred']`) is made **at every timestep**, based on the embedding generated by the words before it, and not only at the last word. This is more efficient than running a new forward propagation from the beginning of the phrase every time a new term should be predicted.

In the code, `layers` is a list holding one dictionary per step. It starts with a layer whose `hidden` vector is `start`. For each word of the sentence, a new layer is created that (1) predicts the word with a softmax over `hidden.dot(decoder)` of the previous layer, (2) adds `-log` of the probability given to the correct word to the `loss`, and (3) computes its own `hidden` state as the previous `hidden` state multiplied by `recurrent` plus the embedding of the word. The function returns the list of layers and the loss.

The list `layers` is a new way to forward propagate. The longer the sentence, the more forward propagation steps are needed, so **static layer variables can no longer be used**; new layers are appended to a list as required. The author stresses that every part of this list should be clear, because otherwise the backpropagation and the weight update will be hard to follow.

In [11]:
def predict(sent):

    layers = list()
    layer = {}
    layer['hidden'] = start
    layers.append(layer)

    loss = 0

    preds = list()
    for target_i in range(len(sent)):

        layer = {}

        layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))

        loss += -np.log(layer['pred'][sent[target_i]])

        layer['hidden'] = layers[-1]['hidden'].dot(recurrent) +\
                                      embed[sent[target_i]]

        layers.append(layer)
    return layers, loss

---
## **16. Backpropagation with Arbitrary Length**

**Backpropagation follows the same logic as described earlier.**

The backward pass applies the "Red Sox defeat Yankees" logic to sequences of any length, using the `layers` list returned by the forward pass. For every layer, a gradient is stored as a new entry: variables such as `sox_delta`, `layer_0_delta`, and `defeat_delta` of the example correspond to the entries of this list. (The book's text names the stored objects `layer['state']`, `layer['previous->hidden']`, and `layer['state_delta']`; the code uses `layer['hidden']` and `layer['hidden_delta']`, and the notebook uses the names of the code.)

The loop walks through the layers in **reverse order**. For a layer that is not the first one, it computes `output_delta` (the prediction minus the one-hot vector of the correct word) and passes it back through the decoder to obtain `new_hidden_delta`. If the layer is the last one, `hidden_delta` is just this value, because **there is no later layer to take a gradient from**; otherwise, the `hidden_delta` of the following layer, carried back through the `recurrent` matrix, is added. For the first layer, `hidden_delta` is only the gradient carried back from the following layer. The cell below runs 30000 iterations of the forward and backward passes with `alpha = 0.001`; it computes the gradients but does not yet change any weight, so it prints nothing.

There are no new concepts in this code, but its construction can look foreign at first. The author recommends being able to read it and explain it to a friend, linking each line to the "Red Sox defeat Yankees" example.

In [12]:
for iter in range(30000):
    alpha = 0.001
    sent = words2indices(tokens[iter%len(tokens)][1:])
    layers,loss = predict(sent)

    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx-1]

        if(layer_idx > 0):
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta']\
                                      .dot(decoder.transpose())

            if(layer_idx == len(layers)-1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + \
                layers[layer_idx+1]['hidden_delta']\
                                      .dot(recurrent.transpose())
        else: # if the first layer
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta']\
                                      .dot(recurrent.transpose())

---
## **17. Weight Update with Arbitrary Length**

**The weights are updated with the same logic as described earlier.**

The weight update is not new either. It is presented after the other two parts so that the reader can concentrate on the **engineering complexity**, having already grasped the theoretical one. The cell below is the complete training loop: forward propagation, backpropagation (as in the previous cell), and then the update of all the weights, followed by a print of the perplexity every 1000 iterations. In the update, the gradients are multiplied by `alpha` and divided by the sentence length, and:

- `start` is updated with the `hidden_delta` of the first layer;
- `decoder` is updated with the outer product of each layer's `hidden` vector and the next layer's `output_delta`;
- the row of `embed` that belongs to each input word is updated with the `hidden_delta` of the corresponding layer;
- `recurrent` is updated with the outer product of each layer's `hidden` vector and the `hidden_delta` computed for the next layer.

Training sentences are taken from `tokens` in turn (`iter%len(tokens)`), and the `[1:]` drops the first word of each sentence. Note that this cell re-defines the same loop as before and adds the update, so it is the one that trains the network.

In [13]:
for iter in range(30000):
    alpha = 0.001
    sent = words2indices(tokens[iter%len(tokens)][1:])

    layers,loss = predict(sent)

    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx-1]

        if(layer_idx > 0):
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta']\
                                      .dot(decoder.transpose())

            if(layer_idx == len(layers)-1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + \
                layers[layer_idx+1]['hidden_delta']\
                                      .dot(recurrent.transpose())
        else:
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta']\
                                      .dot(recurrent.transpose())

    start -= layers[0]['hidden_delta'] * alpha / float(len(sent))
    for layer_idx,layer in enumerate(layers[1:]):

        decoder -= np.outer(layers[layer_idx]['hidden'],\
                        layer['output_delta']) * alpha / float(len(sent))

        embed_idx = sent[layer_idx]
        embed[embed_idx] -= layers[layer_idx]['hidden_delta'] * \
                                        alpha / float(len(sent))
        recurrent -= np.outer(layers[layer_idx]['hidden'],\
                        layer['hidden_delta']) * alpha / float(len(sent))

    if(iter % 1000 == 0):
        print("Perplexity:" + str(np.exp(loss/len(sent))))

Perplexity:81.9504487298333
Perplexity:81.76779959436486
Perplexity:81.52178634167548
Perplexity:81.11266419578483
Perplexity:80.35055497089408
Perplexity:78.7934631759284
Perplexity:75.10917239761613
Perplexity:62.74310959865675
Perplexity:32.3267492987245
Perplexity:20.759331594863056
Perplexity:19.057835050986974
Perplexity:18.105528584386896
Perplexity:17.062591093784242
Perplexity:15.62235152937888
Perplexity:13.360098871984544
Perplexity:10.29280338222749
Perplexity:7.743452303992093
Perplexity:6.487449526010424
Perplexity:5.712373030535116
Perplexity:5.186834079256545
Perplexity:4.8751454717547436
Perplexity:4.6624302083279
Perplexity:4.514684893508487
Perplexity:4.4198471683155915
Perplexity:4.368399599390949
Perplexity:4.3354915587786005
Perplexity:4.297469866329458
Perplexity:4.246067775988757
Perplexity:4.1812197421951565
Perplexity:4.091522941051865


---
## **18. Execution and Output Analysis**

**The moment of truth: what happens when the code runs?**

The author reports a relatively steady **downtrend of the perplexity**. Technically, perplexity is the probability of the correct label (word) passed through a logarithm, negated, and exponentiated ($e^x$). Theoretically, it expresses the **difference between two probability distributions**: the perfect distribution gives 100% probability to the correct term and 0% to everything else. Perplexity is high when the two distributions do not match and low (approaching 1) when they match. A decreasing perplexity, like any loss function used with stochastic gradient descent, is a good sign: the network learns to predict probabilities that match the data.

In the book's run, the printed perplexity starts at about 82.09 (then 81.88 and 81.54) and ends at about 4.13, 4.07, and 4.02.

### **18.1 Looking at the Predictions**

Perplexity says little about what happens inside the weights (and it has been criticized, especially in the language-modeling community, for being overused as a metric). A closer look at the **predictions** is more informative. The cell below takes a sentence of `tokens`, predicts at every position the word the model finds most likely, and prints for each step the previous input, the true next word, and the prediction. This shows what kinds of things the model gets right, what mistakes it makes, and in which order it learns patterns.

In [14]:
sent_index = 4

l,_ = predict(words2indices(tokens[sent_index]))

print(tokens[sent_index])

for i,each_layer in enumerate(l[1:-1]):
    input = tokens[sent_index][i]
    true = tokens[sent_index][i+1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + input + (' ' * (12 - len(input))) +\
          "True:" + true + (" " * (15 - len(true))) + "Pred:" + pred)

['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:is
Prev Input:moved       True:to             Pred:to
Prev Input:to          True:the            Pred:the
Prev Input:the         True:garden.        Pred:bedroom.


### **18.2 What the Network Predicts During Training**

The book prints the output for the sentence "sandra moved to the garden." at three moments of training:

| Previous input | True next word | After 100 steps | After more training (10,000 steps) | After further training |
|---|---|---|---|---|
| sandra | moved | is | the | is |
| moved | to | kitchen | the | to |
| to | the | bedroom | the | the |
| the | garden. | office | the | bedroom. |

- **After 100 steps** the network behaves as random networks do at the start. It is probably just biased toward the words it happened to begin with.
- **After 10,000 steps** it picks out the most common word, "the", and predicts it at **every position**. This is a very common error of recurrent neural networks: learning finer-grained details of a highly skewed dataset takes a lot of training.
- **After further training** the mistakes become interesting. After "sandra" the network predicts "is": not the correct "moved", but not a bad guess either, since it picked the wrong verb. The words "to" and "the" are right, which is unsurprising because they are among the most common words of the dataset and the network has presumably seen "to the" after "moved" many times. The last mistake is also instructive: "bedroom" instead of "garden".

It is important to note that the network can hardly learn this task perfectly. Given only "sandra moved to the", one cannot say with certainty which place comes next; more context is needed. The fact that the task cannot be solved fully makes the analysis of **how it fails** educational.

---
## **19. Summary**

**Recurrent neural networks predict over arbitrary-length sequences.**

The chapter showed how to create vector representations of sequences of arbitrary length. The last exercise trained a **linear recurrent neural network** to predict the next term given the preceding phrase, which required it to learn embeddings that represent variable-length strings of terms in a fixed-size vector.

The key takeaways of Chapter 12 are:

- **Concatenating** word vectors gives vectors of different lengths for different sentences, so comparing them depends on an arbitrary alignment. **Averaging** word vectors gives a fixed length and a surprisingly useful similarity: negative reviews, for example, end up close to the average of "boring" and "awful".
- A word vector can be pictured as a **squiggly line**. Training molds the curves so that certain bumps carry meaning (the negative spike appears only because all negative words have it), and averaging keeps the dominant shapes while single-word noise cancels. A network detects the curves that correlate with a target label.
- Averaging has **two weaknesses**: with too many words the average flattens to near-zero, so information does not decay nicely, and it is a **bag-of-words** method that has **no notion of order** ("Red Sox defeat Yankees" and "Yankees defeat Red Sox" get the same vector, because addition is associative).
- Multiplying by an **identity matrix** between the additions leaves the sentence embedding unchanged. Replacing the identity matrices with **learned transition matrices**, all the same matrix, lets the network learn how word order should shape the sentence embedding, guided by the task of predicting the next word.
- **Backpropagation** is the familiar procedure: when two vectors are added, the same gradient flows into both of them, so the gradient crosses the transition matrix and also reaches the word vector. For sequences of arbitrary length, the forward pass keeps a **list of layers** (one per word, each with its prediction and hidden state), the backward pass walks through that list in reverse, and the update accumulates the weight changes along it.
- Trained on the bAbI toy corpus, the network's **perplexity** fell steadily, from about 82 to about 4. The predictions went from random words to the single most common word ("the"), and then to plausible guesses that still contain interesting mistakes; the task cannot be solved perfectly, since "sandra moved to the" needs more context.
- Sentence vectors do not encode everything in the sentence: a variable amount of information has to fit into a fixed-size box, so the key to RNNs is not only **what these vectors remember but also what they forget**. For next-word prediction, most RNNs learn that only the last couple of words are really necessary and learn to forget (not to create unique patterns for) words further back in the history.
- The representations of this chapter contain **no nonlinearities**. The next chapter explores the limitations this creates, using nonlinearities and gates in the **long short-term memory network (LSTM)**. Before moving on, the reader should be able to code a working linear RNN that converges from memory, because the complexity is about to increase considerably.